In [ ]:
import os
import json
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
# Set a limit to speed up analysis (None = use all test samples)
NUM_SAMPLES = None 
BATCH_SIZE = 20

# Lat/Lon dimensions
lat_dim = 128
num_bins = 64

# Model Architecture Config (Must match training)
architecture = "unet6SR_1x"
base_channels = 8
gn_groups = 8
kernel_size = 3

# Paths
base_dir = Path("/Users/ewellmeyer/Documents/research")
data_dir = base_dir / "HadGEM"
input_file = data_dir / f"GA789_PR_his_rg{lat_dim}.nc"
truth_file = data_dir / f"GA789_dPdK_rg{lat_dim}.nc"
landmask_file = data_dir / "hadgem_landmask_rg128.nc"

# Ensemble Directory
ens_dir = Path(
    f"/Users/ewellmeyer/Documents/research/weights/"
    f"ens_HG789_PR_dPdK_Softmax_{architecture}_ch{base_channels}_k{kernel_size}_{lat_dim}x_dPbins{num_bins}_gn{gn_groups}_sigma64"
)
norm_stats_path = ens_dir / "norm_stats.json"
bin_info_path   = ens_dir / "born_bins.json"
split_ind_path  = ens_dir / "data_splits.npz"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 2. MODEL DEFINITION
# ==============================================================================
from models.unet import Unet6R_1x 

class SoftmaxHead(nn.Module):
    def __init__(self, in_channels, num_bins):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_bins, kernel_size=1)
    def forward_components(self, feat):
        logits = self.conv(feat)
        probs  = torch.softmax(logits, dim=1) 
        return probs 

class ProbUNet(nn.Module):
    def __init__(self, input_channels, base_channels, kernel_size, p_drop, num_bins, gn_groups):
        super().__init__()
        self.backbone = Unet6R_1x(
            input_channels=input_channels,
            output_channels=base_channels,
            base_channels=base_channels,
            kernel_size=kernel_size,
            p_drop=p_drop,
            gn_groups=gn_groups,    
        )
        self.head = SoftmaxHead(base_channels, num_bins)

    def forward_components(self, x):
        feat = self.backbone(x)
        return self.head.forward_components(feat)

# ==============================================================================
# 3. DATA LOADING
# ==============================================================================
def load_data():
    # Load Stats
    with open(norm_stats_path, "r") as f:
        ns = json.load(f)
    x_mean = np.array(ns["x_mean"], dtype=np.float32).reshape(1,1,1,1)
    x_std  = np.array(ns["x_std"],  dtype=np.float32).reshape(1,1,1,1)
    y_mean = float(ns["y_mean"])
    y_std  = float(ns["y_std"])

    # Load Bin Centers
    with open(bin_info_path, "r") as f:
        bi = json.load(f)
    bin_centers = np.array(bi["bin_centers_norm"], dtype=np.float32)

    # Load Splits
    splits = np.load(split_ind_path)
    test_indices = splits["test"] 

    if NUM_SAMPLES is not None:
        # Randomly sample if requesting fewer than full test set
        if NUM_SAMPLES < len(test_indices):
            test_indices = np.random.choice(test_indices, NUM_SAMPLES, replace=False)
            test_indices.sort()
    
    print(f"Evaluating on {len(test_indices)} samples from Test set.")

    # Load NetCDF Data
    ds_in = xr.open_dataset(input_file)
    ds_tar = xr.open_dataset(truth_file)
    
    # Load Landmask
    ds_lm = xr.open_dataset(landmask_file)
    landmask = ds_lm['land_mask'].values
    
    # Extract arrays
    # X: (N, 1, H, W), y: (N, 1, H, W)
    X_full = ds_in.PR.values[:, np.newaxis, ...]
    y_full = ds_tar.dPdK.values[:, np.newaxis, ...]
    
    # Select Test Data
    X_test = X_full[test_indices]
    y_test = y_full[test_indices]
    
    # Normalize
    X_test_norm = (X_test - x_mean) / x_std
    y_test_norm = (y_test - y_mean) / y_std

    return X_test_norm, y_test_norm, bin_centers, landmask

# ==============================================================================
# 4. ENSEMBLE INFERENCE & CDF CALCULATION
# ==============================================================================
def get_predicted_cdfs(X_norm, bin_centers):
    """
    Returns the Cumulative Distribution Function (CDF) for every pixel
    averaged across the ensemble.
    """
    member_files = sorted(list(ens_dir.glob(f"ens_*_member*.pth")))
    if not member_files:
        raise ValueError(f"No checkpoint files found in {ens_dir}")
        
    N, _, H, W = X_norm.shape
    accum_probs = np.zeros((N, num_bins, H, W), dtype=np.float32)
    
    model = ProbUNet(1, base_channels, kernel_size, 0.0, num_bins, gn_groups).to(device)

    print("Running Ensemble Inference...")
    for m_file in tqdm(member_files, desc="Members"):
        ckpt = torch.load(m_file, map_location=device)
        model.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt, strict=False)
        model.eval()

        with torch.no_grad():
            for i in range(0, N, BATCH_SIZE):
                batch_X = torch.tensor(X_norm[i:i+BATCH_SIZE], dtype=torch.float32, device=device)
                probs = model.forward_components(batch_X)
                accum_probs[i:i+BATCH_SIZE] += probs.cpu().numpy()

    # Average Probabilities across members
    avg_probs = accum_probs / len(member_files)
    
    # Calculate CDF by cumulative sum over the bin dimension (axis 1)
    # Shape: (N, num_bins, H, W)
    cdf = np.cumsum(avg_probs, axis=1)
    
    # Ensure it ends exactly at 1.0 (fix potential float errors)
    cdf[:, -1, :, :] = 1.0
    
    return cdf

# ==============================================================================
# 5. CALCULATE RELIABILITY (PIT HISTOGRAM & INTERVAL CALIBRATION)
# ==============================================================================
def calculate_calibration(cdf, y_true_norm, bin_centers, landmask):
    """
    We use the Probability Integral Transform (PIT).
    For a calibrated model, the CDF value at the true observation 
    should be Uniformly Distributed U[0,1].
    """
    print("Calculating Calibration Metrics...")
    
    # Filter to Land Only
    land_indices = np.where(landmask == 1)
    
    # Flatten arrays for land pixels only
    # y_true_norm shape: (N, 1, H, W) -> flatten over N and spatial dims
    y_flat = y_true_norm[:, 0, :, :][:, land_indices[0], land_indices[1]].flatten()
    
    # cdf shape: (N, bins, H, W) -> flatten over N and spatial, keep bins
    cdf_flat = cdf[:, :, land_indices[0], land_indices[1]] # Shape: (N_points, num_bins)
    cdf_flat = cdf_flat.reshape(-1, num_bins) # Ensure 2D
    
    n_points = len(y_flat)
    
    # --- Interpolate CDF to find Probability at True Value (PIT) ---
    # We find the bin indices surrounding the true value y
    idx = np.searchsorted(bin_centers, y_flat)
    
    # Clip to valid range [1, num_bins-1] for interpolation
    idx = np.clip(idx, 1, num_bins - 1)
    
    x0 = bin_centers[idx - 1]
    x1 = bin_centers[idx]
    
    # Get y-values (probabilities) at those indices
    y0 = cdf_flat[np.arange(n_points), idx - 1]
    y1 = cdf_flat[np.arange(n_points), idx]
    
    # Linear Interpolation: pit = y0 + (y_true - x0) * slope
    pit_values = y0 + (y_flat - x0) * (y1 - y0) / (x1 - x0)
    
    # Clip PIT to [0, 1]
    pit_values = np.clip(pit_values, 0.0, 1.0)
    
    return pit_values

# ==============================================================================
# 6. PLOTTING
# ==============================================================================
def plot_reliability(pit_values):
    sns.set_context("paper", font_scale=1.4)
    fig, ax = plt.subplots(1, 2, figsize=(16, 7))

    # --- Plot 1: PIT Histogram (Uniformity Check) ---
    sns.histplot(pit_values, bins=20, stat='density', ax=ax[0], color='skyblue', edgecolor='black', alpha=0.7)
    ax[0].axhline(1.0, color='red', linestyle='--', linewidth=2, label="Perfect Calibration")
    ax[0].set_title("PIT Histogram\n(Flat = Perfect)", fontweight='bold')
    ax[0].set_xlabel("Cumulative Probability at Truth")
    ax[0].set_ylabel("Density")
    ax[0].legend()
    
    # --- Plot 2: Reliability Diagram (Confidence Intervals) ---
    expected_probs = np.linspace(0, 1, 21) # 0%, 5%... 100%
    observed_freqs = []
    
    # For every expected confidence level p (e.g. 90%), we check the 
    # fraction of data falling in the central p-interval centered at 0.5.
    for p in expected_probs:
        lower_q = 0.5 - p/2
        upper_q = 0.5 + p/2
        
        # Check fraction of PIT values inside [lower_q, upper_q]
        count = np.sum((pit_values >= lower_q) & (pit_values <= upper_q))
        observed_freqs.append(count / len(pit_values))
        
    ax[1].plot(expected_probs, observed_freqs, 'o-', color='blue', linewidth=2, label='Model')
    ax[1].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
    
    # MACE (Mean Absolute Calibration Error)
    mace = np.mean(np.abs(np.array(observed_freqs) - expected_probs))
    
    ax[1].set_title(f"Reliability Diagram (Confidence Intervals)\nMACE: {mace:.4f} (Lower is better)", fontweight='bold')
    ax[1].set_xlabel("Predicted Confidence Interval (p)")
    ax[1].set_ylabel("Observed Frequency (Data inside Interval)")
    ax[1].legend()
    ax[1].grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # 1. Load
    X_test, y_test, bin_centers, landmask = load_data()
    
    # 2. Infer
    cdf = get_predicted_cdfs(X_test, bin_centers)
    
    # 3. Calculate
    pit_values = calculate_calibration(cdf, y_test, bin_centers, landmask)
    
    # 4. Plot
    plot_reliability(pit_values)